# IBGE geography: how the raw layer is obtained

**Publisher:** Instituto Brasileiro de Geografia e Estatística (IBGE)  
**Official source:** <https://servicodados.ibge.gov.br/api/docs/localidades>

States and municipalities with their IBGE codes: the key that joins almost every education table.

**Grain and keys:** One row per municipality or state.

**Released as:**

- `ibge-analytics`: semantic (2 tables)
- `ibge-raw-trusted`: raw, trusted (4 tables)

Every table and column is documented in the [interactive data map](https://rangeltech.net/datamap/). Source code and the other notebooks: [https://github.com/LucasRangelSSouza/brazil-public-data-map](https://github.com/LucasRangelSSouza/brazil-public-data-map).

## Status of this notebook

Executed against the live source when this notebook was written; the outputs below are from that run.

In [1]:
import json, time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; public-data-map-notebook; +https://github.com/LucasRangelSSouza/brazil-public-data-map)"}

def get(url, retries=4, timeout=90, **kw):
    """GET with exponential backoff. Public portals answer 429 or reset connections under load."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout, **kw)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

## 1. States and municipalities

The localities API returns the official list with the IBGE codes. The municipality code has seven digits and is the join key used by almost every education table.

In [2]:
BASE = "https://servicodados.ibge.gov.br/api/v1/localidades"
estados = get(f"{BASE}/estados").json()
municipios = get(f"{BASE}/municipios").json()
print(len(estados), "states,", len(municipios), "municipalities")
import pandas as pd
pd.DataFrame([{"codigo_municipio": m["id"], "nome": m["nome"], "uf": m["microrregiao"]["mesorregiao"]["UF"]["sigla"] if m.get("microrregiao") else None} for m in municipios]).head()

27 states, 5571 municipalities


,codigo_municipio,nome,uf
0,1100015,Alta Floresta D'Oeste,RO
1,1100023,Ariquemes,RO
2,1100031,Cabixi,RO
3,1100049,Cacoal,RO
4,1100056,Cerejeiras,RO


## 2. Check the key

A valid municipality code has seven digits. Tables that carry six-digit codes (an older convention) must be completed with the check digit before joining.

In [3]:
assert all(len(str(m["id"])) == 7 for m in municipios)
print("all municipality codes have 7 digits")

all municipality codes have 7 digits


## Compare with the released tables

The released files keep the field names of the source in `raw` and apply consistent typing and snake_case names in `trusted`. Details per column are in the [data map](https://rangeltech.net/datamap/#/subject/ibge).

In [4]:
# Read one released table without downloading the whole dataset. Public datasets need no login.
import kagglehub, pandas as pd
path = kagglehub.dataset_download("lucasrangelss/ibge-raw-trusted", path="trusted__ibge_estados.parquet")
pd.read_parquet(path).head()

C:\Users\lucas.rangel\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0.00/2.35k [00:00<?, ?B/s]

100%|██████████| 2.35k/2.35k [00:00<?, ?B/s]

,codigo_uf,sigla_uf,nome_uf,codigo_regiao,sigla_regiao,nome_regiao
0,50,MS,Mato Grosso do Sul,5,CO,Centro-Oeste
1,52,GO,Goiás,5,CO,Centro-Oeste
2,53,DF,Distrito Federal,5,CO,Centro-Oeste
3,51,MT,Mato Grosso,5,CO,Centro-Oeste
4,24,RN,Rio Grande do Norte,2,NE,Nordeste
